# 07 — Files, Paths, and Serialization (JSON/CSV/Pickle)

Goal: read/write files safely, use pathlib, and serialize data responsibly.

_Generated: 2026-02-19_

## Setup

This course targets **Python 3.11+** (works on 3.10+, with a few feature differences).

Recommended tooling:

```bash
# create + activate a virtual environment
python -m venv .venv
# mac/linux:
source .venv/bin/activate
# windows (PowerShell):
# .venv\Scripts\Activate.ps1

python -m pip install -U pip

# quality-of-life (optional but recommended)
python -m pip install -U ipykernel ruff black pytest mypy
# optional
python -m pip install -U pyyaml
```

If you're using Jupyter:
```bash
python -m ipykernel install --user --name python-course --display-name "Python Course (.venv)"
```

In [ ]:

import sys, platform, os
print("python:", sys.version.split()[0])
print("implementation:", platform.python_implementation())
print("platform:", platform.platform())
print("cwd:", os.getcwd())


## 1.
L1: `pathlib` is the modern filesystem API

Prefer `Path` over string paths.

In [ ]:

from pathlib import Path

root = Path(".")
print("cwd:", root.resolve())
print("py files:", len(list(root.glob("*.py"))))


## 2.
L2: Reading/writing text safely

- Always specify `encoding="utf-8"` for text files.
- Use `with` so files close even on exceptions.

In [ ]:

from pathlib import Path

p = Path("example.txt")
with p.open("w", encoding="utf-8") as f:
    f.write("line1\nline2\n")

with p.open("r", encoding="utf-8") as f:
    for line in f:
        print("read:", line.rstrip("\n"))

p.unlink(missing_ok=True)


## 3.
L3: Streaming large files (don’t `.read()` blindly)

Use iteration or chunk reads for large files.

In [ ]:

from pathlib import Path

p = Path("big.txt")
p.write_text("x\n" * 10_000, encoding="utf-8")

def count_lines(path: Path) -> int:
    n = 0
    with path.open("r", encoding="utf-8") as f:
        for _ in f:
            n += 1
    return n

print("lines:", count_lines(p))
p.unlink(missing_ok=True)


## 4.
L4: JSON (safe, interoperable)

JSON supports:
- dict / list
- str / int / float / bool / null

It does *not* natively support dates/bytes/Decimal (you must convert).

In [ ]:

import json
from pathlib import Path

data = {"name": "Ada", "skills": ["math", "coding"], "active": True}
p = Path("data.json")
p.write_text(json.dumps(data, indent=2), encoding="utf-8")

loaded = json.loads(p.read_text(encoding="utf-8"))
print(loaded)
p.unlink(missing_ok=True)


## 5.
L5: CSV (structured rows)

Use `csv.DictReader` / `csv.DictWriter` for column-based data.

In [ ]:

import csv
from pathlib import Path

rows = [
    {"name": "Ada", "score": 10},
    {"name": "Grace", "score": 12},
]

p = Path("scores.csv")
with p.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["name", "score"])
    w.writeheader()
    w.writerows(rows)

with p.open("r", newline="", encoding="utf-8") as f:
    r = csv.DictReader(f)
    print(list(r))

p.unlink(missing_ok=True)


## 6.
L6: Pickle (powerful but dangerous)

Pickle can execute code when loading untrusted data.
Rule: **never unpickle data you didn’t create**.
Use JSON, msgpack, or a proper DB format instead.

In [ ]:

import pickle
from pathlib import Path

obj = {"a": [1,2,3], "b": ("x", "y")}
p = Path("obj.pkl")
p.write_bytes(pickle.dumps(obj))

loaded = pickle.loads(p.read_bytes())
print(loaded)

p.unlink(missing_ok=True)


## 7.
L7: Temporary files and directories

Use `tempfile` for scratch space; it avoids name collisions and cleans up.

In [ ]:

import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as d:
    dp = Path(d)
    (dp / "hello.txt").write_text("hi", encoding="utf-8")
    print("files:", list(dp.iterdir()))


## 8.
L8: Exercises

1. Write `read_numbers(path)` that returns a list of ints from a file, ignoring blank lines.
2. Write `to_json(path, obj)` and `from_json(path)` helpers.
3. Write a small script that scans a directory and reports counts by file extension.

## 9.
L9: Binary I/O and `io` buffers

Binary files are opened with `"rb"` / `"wb"` and deal in `bytes`.
Useful for images, compressed files, etc.

In [ ]:

from pathlib import Path

p = Path("bin.dat")
p.write_bytes(b"\x00\x01\x02hello")
data = p.read_bytes()
print(data, "len:", len(data))
p.unlink(missing_ok=True)


## 10.
L10: Atomic writes (avoid partial files)

Pattern:
1. write to temp file
2. `replace()` to move into place atomically (on most OSes)

This prevents corrupted files on crashes.

In [ ]:

import tempfile
from pathlib import Path

target = Path("atomic.txt")
with tempfile.NamedTemporaryFile("w", delete=False, encoding="utf-8") as tf:
    tf.write("complete content\n")
    tmp_name = tf.name

Path(tmp_name).replace(target)
print(target.read_text(encoding="utf-8").strip())
target.unlink(missing_ok=True)
